# Fact extraction with Llama 3.1 8B on 5,000 documents (A100 or H100)

**What this tests.** Today the memory bank's typed records come from a rule-based extractor that keeps only some of
the facts people ask about. This notebook asks an open model, Llama 3.1 8B Instruct, to turn every passage of the
5,000-document memory bank into a list of standalone facts. It runs at load time, with no question in mind, and as
fast as one GPU allows (vLLM, batched). It then measures three things.

| Measure | How | Needs |
|---|---|---|
| **Answer facts kept** | The benchmark's answer facts are checked against the extracted facts of their gold documents, by the evidence audit's word check, and against today's rule-based records | nothing |
| **Correctness** | Numbers not in the source passage, duplicates. On samples, a judge model checks whether answer facts survive and whether extracted facts are correct | the judge: `OPENAI_API_KEY` |
| **Speed** | Output tokens per second, passages per second, and the hours this projects to for 5,000, 50,000 and 512,000 documents | nothing |

What decides the result is fixed in advance in `docs/FACT_EXTRACTION_PREREGISTRATION.md`, and for the fast setting in
`docs/FACT_EXTRACTION_FAST_PREREGISTRATION.md`. The last cells check those criteria. Run 1's results are in
`docs/FACT_EXTRACTION_RESULTS.md`.

**Two settings** (`SPEED` in the first cell)
- **fast** (the default) makes three changes:
  - the model writes shorter facts, with no tag line and no subject repeated for every item of a list;
  - each passage leaves out its start that repeats the previous passage (about 10% of the text);
  - the weights are 8-bit: INT8 on an A100, FP8 on an L4 or H100.

  Writing the output is most of the GPU time, so this is expected to be about twice as fast on an A100. That is an
  estimate, and this run measures it. The same checks decide whether the shorter facts keep the answer facts.
- **as run 1**: bf16 weights, the full instructions and whole passages, about 50 minutes on an A100 80 GB.
- **both** runs the two on this GPU, one after the other, for a like-for-like comparison. With `LIMIT_PASSAGES = 2000`
  this is a quick speed check (about 10 minutes); a later full run continues the same files.

**Before you start**
1. Runtime > Change runtime type > **H100 GPU** if your plan offers it (the fastest), otherwise **A100 GPU** (40 or 80 GB).
2. Secrets (the key icon in the left bar):
   - `OPENAI_API_KEY`, for the judge (gpt-5.4-mini, about USD 1);
   - optionally `HF_TOKEN` with access to `meta-llama/Llama-3.1-8B-Instruct`. Without it, the identical copy
     `RedHatAI/Llama-3.1-8B-Instruct` is used.
3. Runtime > Run all, and allow Google Drive when asked. The results are written to Drive as they come
   (`MyDrive/cie_fact_extraction`), so a disconnected runtime loses nothing. Run all again and it carries on where it
   stopped.

**What it takes**, measured on an A100 80 GB in run 1:
- installing, a few minutes;
- getting the benchmark (about 5 GB), a few minutes;
- building the 33,363 passages, a few minutes (once: they are kept on Drive);
- loading the model, about 2.5 minutes;
- the extraction "as run 1", about 50 minutes (3,800 output tokens and 11 passages per second); "fast", not measured
  yet;
- the judge, a few minutes.

A 40 GB A100 is slower. `LIMIT_PASSAGES = 2000` gives a quick look first.

In [ ]:
#@title Parameters
import os
REPO = "https://github.com/Nikku03/integratedai"
BRANCH = "claude/epic-keller-gevn7j"
SPEED = "fast"                 #@param ["fast", "as run 1", "both"]
MODELS = "llama-3.1-8b"        #@param ["llama-3.1-8b", "llama-3.1-8b,llama-3.2-3b", "llama-3.2-3b"] {allow-input: true}
DOCS = 5000                    #@param {type:"integer"}
LIMIT_PASSAGES = 0             #@param {type:"integer"}
MAX_TOKENS = 768               #@param {type:"integer"}
JUDGE = True                   #@param {type:"boolean"}
JUDGE_MODEL = "gpt-5.4-mini"   #@param ["gpt-5.4-mini", "gpt-5.4", "gpt-5-mini"] {allow-input: true}
JUDGE_FACTS = 300              #@param {type:"integer"}
JUDGE_LINES = 300              #@param {type:"integer"}
MAX_BATCHED_TOKENS = 0         #@param {type:"integer"}
SAVE_TO_DRIVE = True           #@param {type:"boolean"}
# SPEED: "fast" = shorter facts, repeated text left out, 8-bit weights for this GPU (INT8 on an A100, FP8 on an L4 or
#   H100). "as run 1" = bf16 weights, the full instructions, whole passages. "both" = one after the other.
# MODELS: llama-3.1-8b is the model under test. Add llama-3.2-3b to compare; each model runs in its own process, one after the other.
# DOCS 5000: the 5,000-document memory bank's documents (every gold document plus a sample stratified by source, seed 5).
# LIMIT_PASSAGES: 0 = every passage (33,363 for 5,000 documents). 2000 gives a quick look in a few minutes; the criteria
#   are only decided on a full run.
# MAX_TOKENS: the most one passage's fact list may use (768 is the pre-registered setting); passages that reach it are
#   counted in the report.
# JUDGE: JUDGE_MODEL checks JUDGE_FACTS sampled answer facts and JUDGE_LINES sampled fact lines per model. It needs the
#   OPENAI_API_KEY secret. The pre-registered criteria 1 and 2 need it.
# MAX_BATCHED_TOKENS: 0 picks by GPU: 8192 on 40 GB, 16384 on 80 GB (run 1: 11.4 passages/s). vLLM's own default on any A100 is 8192.
# SAVE_TO_DRIVE: results go to MyDrive/cie_fact_extraction as they are written, so a disconnect loses nothing and a
#   new runtime continues the run. Off: they stay on this runtime's disk and are lost with it.
# Each setting writes its own facts file (facts_llama-3.1-8b.jsonl, facts_llama-3.1-8b-fast-int8.jsonl, ...), so the
#   report shows them side by side. Changing MAX_TOKENS after a run: the extraction refuses to mix it into the old
#   facts file; delete that facts_*.jsonl in the results folder to start it again.
JUDGE_WANTED = JUDGE
ON_COLAB = os.path.isdir("/content")
WORKDIR = "/content/integratedai/cie" if ON_COLAB else os.environ.get("CIE_COLAB_WORKDIR", os.getcwd())
BENCH = "/content/EnterpriseRAG-Bench-json" if ON_COLAB else os.environ.get("CIE_ERB_ROOT", os.path.join(os.getcwd(), "EnterpriseRAG-Bench"))
# verification runs without a GPU: an OpenAI-compatible server (such as Ollama) stands in for vLLM
BACKEND = os.environ.get("EXTRACT_BACKEND", "vllm")
BASE_URL = os.environ.get("EXTRACT_BASE_URL", "")
MODELS = os.environ.get("EXTRACT_MODELS", MODELS)
SPEED = os.environ.get("EXTRACT_SPEED", SPEED)
DOCS = int(os.environ.get("CIE_ERB_DOCS", DOCS))
LIMIT_PASSAGES = int(os.environ.get("EXTRACT_LIMIT", LIMIT_PASSAGES))
QUESTIONS = int(os.environ.get("EXTRACT_QUESTIONS", "0"))
JUDGE_WANTED = JUDGE_WANTED and os.environ.get("EXTRACT_JUDGE", "1") != "0"
MODEL_LIST = [m.strip() for m in MODELS.split(",") if m.strip()]
RUN_NAME = f"extract_facts_{DOCS}" + (f"_q{QUESTIONS}" if QUESTIONS else "")
print("setting:", SPEED, "| models:", MODEL_LIST, "| documents:", DOCS, "| passages:", LIMIT_PASSAGES or "all", "| output cap:", MAX_TOKENS,
      "| judge:", JUDGE_MODEL if JUDGE_WANTED else "off")
print("backend:", BACKEND, "| colab:", ON_COLAB, "| results to Drive:", SAVE_TO_DRIVE and ON_COLAB)

In [ ]:
#@title GPU check and the runs for this GPU (H100, or an A100 of 40 or 80 GB)
import os, subprocess
q = subprocess.run("nvidia-smi --query-gpu=name,memory.total,driver_version,compute_cap --format=csv,noheader,nounits", shell=True,
                   capture_output=True, text=True).stdout.strip().splitlines()
print(q[0] if q else "no GPU visible")
GPU_NAME, GPU_MEM_GB, DRIVER_MAJOR, CC = "", 0.0, 0, 0.0
if q:
    f = [x.strip() for x in q[0].split(",")]
    GPU_NAME, GPU_MEM_GB, DRIVER_MAJOR, CC = f[0], int(f[1]) / 1024, int(f[2].split(".")[0]), float(f[3])  # memory.total is in MiB
if BACKEND == "vllm":
    if not q:
        raise SystemExit("this notebook needs a GPU: Runtime > Change runtime type > H100 or A100 GPU, then run all again")
    if CC < 8.0:
        raise SystemExit(f"{GPU_NAME} (compute capability {CC}) cannot run the model in bf16: choose an H100 or A100 "
                         "(Runtime > Change runtime type), then run all again")
# requests in flight: an 8B model in bf16 takes about 15 GB (8-bit: about 9 GB); the rest of the GPU holds the KV cache
MAX_NUM_SEQS = 512 if GPU_MEM_GB >= 70 else 256 if GPU_MEM_GB >= 35 else 128
BATCHED_TOKENS = MAX_BATCHED_TOKENS or (16384 if GPU_MEM_GB >= 70 else 8192)
# vLLM's default build targets CUDA 13, which needs driver 580 or newer; an older driver gets the CUDA 12.9 build
OLD_DRIVER = bool(q) and DRIVER_MAJOR < 580
if q and not any(g in GPU_NAME for g in ("A100", "H100")):
    print("note: not an A100 or H100; any GPU with 24 GB or more and compute capability 8.0 or higher works, at its own speed")
print(f"GPU memory {GPU_MEM_GB:.0f} GB -> up to {MAX_NUM_SEQS} passages in flight, {BATCHED_TOKENS} tokens per step"
      + (f" | driver {DRIVER_MAJOR}: the CUDA 12.9 build of vLLM will be installed" if OLD_DRIVER else ""))
# 8-bit weights for this GPU: FP8 runs natively from compute capability 8.9 (L4, H100); the A100 (8.0) runs INT8
EIGHT_BIT = "fp8" if CC >= 8.9 else "int8"
RUNS = []  # (facts file name, model, extra arguments)
for m in MODEL_LIST:
    if SPEED in ("as run 1", "both"):
        RUNS.append((m.replace(":", "-"), m, []))
    if SPEED in ("fast", "both"):
        quantized = f"{m}-{EIGHT_BIT}" if m == "llama-3.1-8b" else m  # 8-bit copies are pinned for the 8B model only
        RUNS.append((m.replace(":", "-") + ("-fast-" + EIGHT_BIT if quantized != m else "-fast"), quantized,
                     ["--prompt", "short", "--strip-overlap"]))
for name, model, extra in RUNS:
    print(f"run: facts_{name}.jsonl <- {model} {' '.join(extra)}")

In [ ]:
#@title Clone the repository and install vLLM (a few minutes)
import importlib, os, subprocess, sys
if ON_COLAB and not os.path.isdir(WORKDIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, "/content/integratedai"], check=True)
elif ON_COLAB:
    # an existing checkout moves to the latest commit of the branch: a re-run never tests stale code
    subprocess.run(["git", "fetch", "--depth", "1", "origin", BRANCH], cwd="/content/integratedai", check=True)
    subprocess.run(["git", "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"], cwd="/content/integratedai", check=True)
print("code at commit:", subprocess.run(["git", "log", "-1", "--format=%h %ci %s"], cwd=WORKDIR, capture_output=True, text=True).stdout.strip())
if ON_COLAB and BACKEND == "vllm":
    # vLLM brings its own torch; this kernel never imports torch, so no restart is needed
    if OLD_DRIVER:
        spec = ["vllm==0.30.0+cu129", "torch==2.13.0+cu129", "--extra-index-url", "https://wheels.vllm.ai/0.30.0/cu129",
                "--extra-index-url", "https://download.pytorch.org/whl/cu129"]
    else:
        spec = ["vllm==0.30.0"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *spec, "openai>=1.40"], check=True)
if ON_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", WORKDIR], check=True)  # without optional extras: no second torch
src = os.path.join(WORKDIR, "src")
if src not in sys.path:
    sys.path.insert(0, src)
importlib.invalidate_caches()
os.chdir(WORKDIR)
if BACKEND == "vllm":
    # checked in a fresh process, so the extraction processes get a clean GPU
    r = subprocess.run([sys.executable, "-c", "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, "
                        "'| cuda', torch.version.cuda, '| GPU', torch.cuda.get_device_name(0))"], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr[-1500:])
    assert r.returncode == 0, "vLLM did not import (output above)"
import cie
print("cie", cie.__version__, "from", os.path.dirname(cie.__file__))

In [ ]:
#@title Keys from Colab secrets (never printed)
import os

def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:  # not on Colab, or no such secret
        return os.environ.get(name)

for name in ("HF_TOKEN", "OPENAI_API_KEY"):
    v = secret(name)
    if v:
        os.environ[name] = v
JUDGE = JUDGE_WANTED and bool(os.environ.get("OPENAI_API_KEY"))  # re-running this cell after adding the key turns the judge on
print("HF_TOKEN:", "set" if os.environ.get("HF_TOKEN") else "not set (the identical RedHatAI copy of the weights is used)")
print("OPENAI_API_KEY:", "set" if os.environ.get("OPENAI_API_KEY") else "not set")
if JUDGE_WANTED and not JUDGE:
    print("the judge is off: add the OPENAI_API_KEY secret and run this cell again (criteria 1 and 2 need the judge)")

In [ ]:
#@title Results folder (Google Drive: a disconnect loses nothing)
import os
WORK = os.path.join(WORKDIR, "eval_out", RUN_NAME)
if SAVE_TO_DRIVE and ON_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        WORK = os.path.join("/content/drive/MyDrive/cie_fact_extraction", RUN_NAME)
    except Exception as e:  # declined or unavailable
        print(f"Google Drive is not available ({type(e).__name__}): results stay on this runtime's disk and are lost with it")
os.makedirs(WORK, exist_ok=True)
have = sorted(f for f in os.listdir(WORK) if f.startswith("facts_") and f.endswith(".jsonl"))
print("results:", WORK)
print("found from an earlier run:", ", ".join(have) if have else "nothing (a fresh run)")

In [ ]:
#@title Get the benchmark: the repository's JSON records (about 5 GB, a few minutes)
import os, shutil, subprocess
sources = os.path.join(BENCH, "generated_data", "sources")

def _complete():
    """Every file the clone should hold is on disk (an interrupted clone or checkout is repaired or redone)."""
    if not os.path.isfile(os.path.join(BENCH, "questions.jsonl")) or not os.path.isdir(os.path.join(BENCH, ".git")):
        return False
    listed = subprocess.run(["git", "ls-files", "generated_data/sources"], cwd=BENCH, capture_output=True, text=True)
    want = sum(1 for f in listed.stdout.splitlines() if f.endswith(".json"))
    have = sum(1 for _, _, fs in os.walk(sources) for f in fs if f.endswith(".json"))
    print(f"benchmark checkout: {have:,} of {want:,} JSON records on disk")
    return listed.returncode == 0 and want > 1000 and have == want

ok = _complete()
if not ok and os.path.isdir(os.path.join(BENCH, ".git")):
    subprocess.run(["git", "checkout", "-f", "HEAD", "--", "."], cwd=BENCH)  # restores files an interrupted checkout left out
    ok = _complete()
if not ok:
    shutil.rmtree(BENCH, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/onyx-dot-app/EnterpriseRAG-Bench", BENCH], check=True)
    ok = _complete()
assert ok, "the benchmark clone is incomplete; run this cell again"
print("questions:", sum(1 for _ in open(os.path.join(BENCH, "questions.jsonl"))))

In [ ]:
#@title Build the documents and their passages (no model; the first run reads the whole corpus once for the index)
import json, os, subprocess, sys

def run(args, env=None):
    """Runs a step in its own process and shows its output as it comes."""
    print("$", " ".join(args), flush=True)
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, cwd=WORKDIR)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise SystemExit(f"step failed with exit code {p.returncode} (output above)")

def env():
    """The current environment for a step (read at each call, so keys added later reach it)."""
    return {**os.environ, "PYTHONUNBUFFERED": "1", "TQDM_MININTERVAL": "20", "TOKENIZERS_PARALLELISM": "false"}

set_file = os.path.join(WORK, "set.json")
st = json.load(open(set_file)) if os.path.exists(set_file) else {}
if st.get("n_docs") == DOCS and os.path.exists(os.path.join(WORK, "passages.jsonl")):
    print("reusing the passages built earlier (same documents: the selection is fixed by seed 5)")
else:
    run([sys.executable, "-u", "-m", "cie.eval.extract_facts", "build", "--root", BENCH, "--work", WORK, "--docs", str(DOCS)]
        + (["--questions", str(QUESTIONS)] if QUESTIONS else []), env=env())
    st = json.load(open(set_file))
print(f"{st['documents']:,} documents ({st['gold_documents']} gold), {st['passages']:,} passages ({st['passage_chars']:,} characters), "
      f"{st['records']:,} rule-based records; {st['failed']} documents failed to build")

In [ ]:
#@title Extract the facts: each run in its own process (vLLM, batched; saved after every 8,192 passages)
import json, os, sys
for name, m, extra in RUNS:
    args = [sys.executable, "-u", "-m", "cie.eval.extract_facts", "run", "--work", WORK, "--model", m, "--max-tokens", str(MAX_TOKENS),
            "--name", name, "--chunk", "8192", *extra]
    if BACKEND == "vllm":
        args += ["--max-num-seqs", str(MAX_NUM_SEQS), "--max-num-batched-tokens", str(BATCHED_TOKENS), "--gpu-mem", "0.92"]
    else:
        args += ["--backend", "openai", "--base-url", BASE_URL, "--concurrency", "1"]
    if LIMIT_PASSAGES:
        args += ["--limit", str(LIMIT_PASSAGES)]
    run(args, env=env())
    meta = json.load(open(os.path.join(WORK, f"facts_{name}.run.json")))
    s = meta.get("summary") or {}
    if s.get("passages"):
        print(f"\n{name} ({meta.get('model_repo')}): {s['passages']:,} passages in {s['generation_seconds']:,.0f} s "
              f"| {s['output_tokens_per_second']:,} output tokens/s | {s['passages_per_second']} passages/s "
              f"| prompt tokens from cache {s.get('prompt_tokens_from_cache')} | model load {s.get('model_load_seconds')} s\n")

In [ ]:
#@title Score: answer facts kept, numbers grounded, duplicates, size and speed (no model, a minute or two)
import os, sys
from IPython.display import Markdown, display
run([sys.executable, "-u", "-m", "cie.eval.extract_facts", "score", "--work", WORK, "--root", BENCH, "--quiet"], env=env())
display(Markdown(open(os.path.join(WORK, "report.md")).read()))

In [ ]:
#@title Judge samples with an OpenAI model (answer facts kept; extracted facts correct)
import json, os, sys
from IPython.display import Markdown, display
if not JUDGE:
    print("the judge is off (JUDGE, or no OPENAI_API_KEY secret)")
else:
    run([sys.executable, "-u", "-m", "cie.eval.extract_facts", "judge", "--work", WORK, "--root", BENCH, "--model", JUDGE_MODEL,
         "--retention", str(JUDGE_FACTS), "--lines", str(JUDGE_LINES), "--quiet"], env=env())
    from cie.eval.extract_facts import judge_markdown
    display(Markdown(judge_markdown(json.load(open(os.path.join(WORK, "judge.json"))))))

In [ ]:
#@title Pre-registered criteria (docs/FACT_EXTRACTION_PREREGISTRATION.md; the fast setting: docs/FACT_EXTRACTION_FAST_PREREGISTRATION.md)
import json, os
from cie.eval.extract_facts import BASELINE, CARD_FREE
rep = json.load(open(os.path.join(WORK, "report.json")))
jpath = os.path.join(WORK, "judge.json")
jd = json.load(open(jpath)) if os.path.exists(jpath) else None  # the score step removes a judge of earlier facts
# a judge with more than 5% failed calls leaves criteria 1 and 2 undecided (pre-registered)
judge_ok = bool(jd) and jd.get("calls", 0) > 0 and (jd.get("failed_share") or 0) <= 0.05
if jd and not judge_ok:
    print(f"the judge failed on {jd.get('judge_failed')} of {jd.get('calls')} calls: criteria 1 and 2 stay undecided\n")
ret_all = (jd or {}).get("retention", {})
rb = ret_all.get(BASELINE)
show = lambda c: "undecided" if c is None else ("met" if c else "NOT met")
combine = lambda cs: False if False in cs else None if None in cs else True
# run 1 (docs/FACT_EXTRACTION_RESULTS.md): the reference when the run-1 setting has no full run in this folder
RUN_1 = {"passages_per_second": 11.36, "kept_share": round(1215 / 1388, 3)}
three = {}
for name, mm in rep["methods"].items():
    full = mm["passages_done"] == mm["passage_count"]
    num_share = mm["lines_with_a_number_not_in_the_passage"] / max(1, mm["fact_lines"])
    ret, cor = ret_all.get(name), (jd or {}).get("correctness", {}).get(name)
    c1 = c2 = None
    if judge_ok and ret and rb and ret["good_share"] is not None and rb["good_share"] is not None:
        c1 = ret["good_share"] >= 0.80 and round(ret["good_share"] - rb["good_share"], 6) >= 0.15
    if judge_ok and cor and cor["bad_share"] is not None:
        c2 = cor["bad_share"] <= 0.05 and cor["bad_ci95"][1] <= 0.08
    c3 = num_share <= 0.02
    under_test = name.startswith("llama-3.1-8b")
    print(f"== {name}" + ("" if under_test else "  (for information)")
          + ("" if full else f"  (only {mm['passages_done']:,} of {mm['passage_count']:,} passages extracted: not a decision)"))
    if c1 is not None:
        print(f"  1. answer facts covered {ret['good_share']:.0%} (rule-based records {rb['good_share']:.0%}): need >= 80% and +15 points -> {show(c1)}")
    else:
        print(f"  1. answer facts covered: {show(c1)} (needs a usable judge)")
    if c2 is not None:
        print(f"  2. wrong or unsupported {cor['bad_share']:.1%} (95% up to {cor['bad_ci95'][1]:.1%}): need <= 5% and <= 8% -> {show(c2)}")
    else:
        print(f"  2. wrong or unsupported: {show(c2)} (needs a usable judge)")
    print(f"  3. fact lines with a number not in the passage {num_share:.1%}: need <= 2% -> {show(c3)}")
    if under_test:
        three[name] = combine((c1, c2, c3)) if full else None
        verdict = {False: "not met: the rule-based records stay", None: "undecided: criteria 1 and 2 need a usable judge",
                   True: "all three met: next, load these facts as memory records (separately pre-registered)"}[combine((c1, c2, c3))]
        print("  decision:", verdict, "" if full else "(not a full run: not a decision)")
    print()
# the fast setting: the three criteria, plus answer facts kept and speed against the run-1 setting
base = rep["methods"].get("llama-3.1-8b")
base_full = bool(base) and base["passages_done"] == base["passage_count"]
for name, mm in rep["methods"].items():
    if not name.startswith("llama-3.1-8b-fast"):
        continue
    k = rep["kept"][name]
    if base_full:
        ref, ref_from = rep["kept"]["llama-3.1-8b"]["share_without_tag_lines"], "the run-1 setting in this folder, without tag lines"
    else:
        ref, ref_from = RUN_1["kept_share"], "run 1, with its tag lines"
    f2 = round(k["share_without_tag_lines"] - ref, 6) >= -0.03
    sp = (mm["speed"] or {}).get("passages_per_second")
    gpu = (mm["speed"] or {}).get("gpu") or ""
    bsp = (base["speed"] or {}) if base_full else {}
    if bsp.get("passages_per_second") and bsp.get("gpu") == gpu:
        sref, sfrom = bsp["passages_per_second"], "the run-1 setting on this GPU"
    elif "A100" in gpu and "80GB" in gpu.replace(" ", ""):
        sref, sfrom = RUN_1["passages_per_second"], "run 1 on an A100 80 GB"
    else:
        sref, sfrom = None, "no run-1 setting measured on this GPU type"
    f3 = round(sp / sref, 6) >= 1.5 if sp and sref else None
    full = mm["passages_done"] == mm["passage_count"]
    f1 = three.get(name)
    print(f"== {name}: the fast setting" + ("" if full else " (not a full run: not a decision)"))
    print(f"  F1. criteria 1-3 above -> {show(f1)}")
    print(f"  F2. answer facts kept (word check, without tag lines) {k['share_without_tag_lines']:.1%}, against {ref:.1%} ({ref_from}): "
          f"at most 3 points lower -> {show(f2)}")
    print(f"  F3. speed {sp} passages/s, against {sref if sref else '-'} ({sfrom}): at least 1.5 times -> {show(f3)}"
          + (f" ({sp / sref:.2f} times)" if sp and sref else ""))
    fast = combine((f1, f2, f3)) if full else None
    print("  decision:", {False: "not met: the run-1 setting stays for large loads",
                          None: "undecided (see the lines above)",
                          True: "met: the fast setting is used for large loads"}[fast])
cf = ret_all.get(CARD_FREE)
if cf and cf["good_share"] is not None:
    print(f"\nfor information: {CARD_FREE}: {cf['good_share']:.0%} covered (the card's copied passages left out)")

In [ ]:
#@title Look at a few passages and their facts
import glob, json, os, random
from cie.eval.extract_facts import load_facts, load_jsonl
passages = {p["id"]: p for p in load_jsonl(os.path.join(WORK, "passages.jsonl"))}
facts = {os.path.basename(f)[6:-6]: {r["id"]: r for r in load_facts(f)} for f in sorted(glob.glob(os.path.join(WORK, "facts_*.jsonl")))}
common = sorted(set.intersection(*(set(f) for f in facts.values()))) if facts else []
for pid in random.Random(1).sample(common, min(3, len(common))):
    p = passages[pid]
    print("=" * 100, f"\n{p['doc_title'][:90]}  [{p['source']}]\n{p['title']}\n{p['text'][:1200]}\n")
    for m, by in facts.items():
        print(f"--- {m}: {len(by[pid]['facts'])} facts ({by[pid]['output_tokens']} tokens, {by[pid]['finish_reason']})")
        for f in by[pid]["facts"][:25]:
            print("  -", f)
    print()

In [ ]:
#@title Where the results are
import os
print(f"Results: {WORK}" + (" (Google Drive)" if WORK.startswith("/content/drive") else " (this runtime's disk: lost when it is recycled)"))
for f in sorted(os.listdir(WORK)):
    if f != "index.json":
        print(f"  {f:40s} {os.path.getsize(os.path.join(WORK, f)) / 1e6:8.1f} MB")
print("Paste report.md, the judge table and the criteria above into the chat to have them recorded.")